# Inelastic dark matter

Inelastic dark matter models assume that scattering changes the internal state of the dark matter particle.

The mass splitting

$$
\delta = m_\chi^\prime - m_\chi
$$

modifies the recoil kinematics and can dramatically alter detector sensitivity.

This notebook illustrates several benchmark scenarios relevant for xenon and argon experiments.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.nuclei.isotopes import Xe131, Ar40
from src.halo import StandardHaloModel
from src.interactions.si import SIInteraction
from src.interactions.particles import WIMP
from src.nuclei.form_factors import HelmFormFactor
from src.detectors.detectors import IdealDetector
from src.rates import RateCalculator
from src.utils.factory import create_kinematics

from src.plotting.plots_rates import plot_recoil_spectrum

import numpy as np
import matplotlib.pyplot as plt

## Xenon benchmark

Heavy xenon nuclei are particularly effective for probing inelastic dark matter because their large nuclear mass reduces the kinematic penalty associated with the mass splitting.

The recoil spectra below qualitatively reproduce the behaviour commonly observed in LZ-style analyses.

In [ ]:
Egrid = np.linspace(1, 3.5e5, 100)

shm = StandardHaloModel().boost()

wimp=WIMP(mass=1e12, spin=0)
target=Xe131
helm = HelmFormFactor(nucleus=Xe131)

delta = [0, 2e5, 3e5]
xe_rates = []

for d in delta:

    kinematics = create_kinematics(wimp=wimp, nucleus=Xe131, delta=d)
    si = SIInteraction(wimp=wimp, nucleus=target,
                       kinematics=kinematics,
                       form_factor=helm,
                       sigma_p=2e-41)
    
    detector = IdealDetector(nucleus=target, mass_kg=1e6,
                             exposure_days=365)

    xe_rate = RateCalculator(wimp=wimp, halo=shm, interaction=si,
                             detector=detector)
    xe_rates.append(xe_rate)

fig, ax = plot_recoil_spectrum(rate_calc=xe_rates[0], ER=Egrid, label=r"$\delta$ = 0")
fig, ax = plot_recoil_spectrum(rate_calc=xe_rates[1], ER=Egrid, 
                                          label=r"$\delta$ = 200 keV", ax=ax)
fig, ax = plot_recoil_spectrum(rate_calc=xe_rates[2], ER=Egrid, 
                                label=r"$\delta$ = 300 keV", ax=ax)

ax.set_ylabel("Differential rate [/tonne/year/eV]")

ax.legend()
plt.show()

## Argon recoil spectrum

Argon nuclei have an higher velocity threshold for the same dark matter mass splitting.

The plot below illustrates how the different mass splittings affect the accessible recoil-energy spectrum. For a mass splitting of 200 keV, no WIMP in the Standard Halo Model can produce an inelastic recoil.

In [ ]:
Egrid = np.linspace(1, 3.5e5, 100)

wimp=WIMP(mass=200e9, spin=0)
target=Ar40
helm = HelmFormFactor(nucleus=target)

delta = [0, 1e5, 2e5]
ar_rates = []

for d in delta:

    kinematics = create_kinematics(wimp=wimp, nucleus=target, delta=d)
    si = SIInteraction(wimp=wimp, nucleus=target,
                       kinematics=kinematics,
                       form_factor=helm,
                       sigma_p=2e-41)
    
    detector = IdealDetector(nucleus=target, mass_kg=1e6,
                             exposure_days=365)

    ar_rate = RateCalculator(wimp=wimp, halo=shm, interaction=si,
                             detector=detector)
    ar_rates.append(ar_rate)

fig, ax = plot_recoil_spectrum(rate_calc=ar_rates[0], ER=Egrid, label=r"$\delta$ = 0")
fig, ax = plot_recoil_spectrum(rate_calc=ar_rates[1], ER=Egrid, 
                                          label=r"$\delta$ = 100 keV", ax=ax)
fig, ax = plot_recoil_spectrum(rate_calc=ar_rates[2], ER=Egrid, 
                                label=r"$\delta$ = 200 keV", ax=ax)

ax.set_ylabel("Differential rate [/tonne/year/eV]")

ax.legend()
plt.show()